# 🖥️ เทรน G1 หยิบของบนโต๊ะ (Lift-Cube) บน NVIDIA DGX Spark (GB10)

notebook นี้รันบน **Jupyter ของ DGX Spark** (ARM + Blackwell GPU) โดยตรง
— ไม่ต้อง mount Drive, ใช้ storage ของเครื่อง, ไม่มี timeout

**Lift-Cube คืออะไร:** หุ่น G1 ถูกยึดที่เอว (fixed-base) ขยับแค่แขน+มือ Dex3
เพื่อเอื้อมไปหยิบลูกบาศก์บนโต๊ะแล้วยกขึ้นตามความสูงที่สั่ง งานนี้ **ไม่มีปัญหา
ล้ม/ทรงตัว** จึงเรียนง่ายกว่า loco-manip มาก — เป็น Phase 1 ของเส้นทางสู่ VLA
(ดู `docs/development/g1_manipulation_vla.md`) พอสำเร็จค่อยต่อยอดเป็น loco-manip

> ✅ **ทดสอบแล้วเทรนได้จริงบน GB10** (smoke test ผ่าน)
>
> GB10 เป็น Blackwell รุ่นใหม่ (sm_121) ใหม่กว่าที่ nvrtc ของ warp/torch รู้จัก
> (รองรับสูงสุด sm_120) จึงต้องมี 2 workaround ที่ notebook ตั้งให้อัตโนมัติ:
> 1. `wp.config.ptx_target_arch=120` (ผ่าน sitecustomize) — warp สร้าง PTX sm_120
>    แล้ว driver 13.0 JIT ต่อเป็น sm_121
> 2. `PYTORCH_JIT=0` — ปิด torch JIT fusion ที่พยายาม compile ตรงเป็น sm_121

## 1) ตั้งค่า + เช็คเครื่อง

In [ ]:
import os, subprocess, sys
WORKSPACE = '/home/nexpie/workspace/anun'
REPO = os.path.join(WORKSPACE, 'mjlab-custom')
UV = os.path.expanduser('~/.local/bin/uv')
print('arch:', os.uname().machine)  # aarch64
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2) ติดตั้ง uv (ถ้ายังไม่มี)

In [ ]:
import shutil
if not os.path.exists(UV):
    !curl -LsSf https://astral.sh/uv/install.sh | sh
print('uv:', UV, '->', os.path.exists(UV))

## 3) clone repo (ถ้ายังไม่มี)

In [ ]:
os.makedirs(WORKSPACE, exist_ok=True)
if not os.path.isdir(REPO):
    !cd {WORKSPACE} && git clone -q https://github.com/anunpanya9/mjlab-custom.git
else:
    !cd {REPO} && git pull -q
print('repo:', REPO)

## 4) แก้ pyproject ให้รองรับ ARM (aarch64)

mjlab ตั้ง required-environments เป็น linux x86_64 เท่านั้น — เพิ่ม aarch64
ให้ uv ยอมติดตั้งบน DGX Spark (แก้เฉพาะบนเครื่องนี้ ไม่ push)

In [ ]:
pp = os.path.join(REPO, 'pyproject.toml')
txt = open(pp).read()
arm_line = '  "sys_platform == \'linux\' and platform_machine == \'aarch64\'",'
if 'aarch64' not in txt:
    txt = txt.replace(
        '  "sys_platform == \'linux\' and platform_machine == \'x86_64\'",',
        '  "sys_platform == \'linux\' and platform_machine == \'x86_64\'",\n' + arm_line)
    open(pp,'w').write(txt)
    print('เพิ่ม aarch64 แล้ว')
else:
    print('มี aarch64 อยู่แล้ว')

## 5) uv sync + ติดตั้ง GB10 workaround

ครั้งแรกดาวน์โหลดหลาย GB ใช้เวลาสักครู่ หลัง sync จะติดตั้ง sitecustomize
(ptx_target_arch=120) ให้อัตโนมัติ

In [ ]:
!cd {REPO} && {UV} sync --extra cu128 2>&1 | tail -15
print('=== เช็ค torch เห็น GPU ไหม ===')
!cd {REPO} && {UV} run --no-sync python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')"

# --- GB10 (sm_121) workaround ---
# GB10 เป็น Blackwell รุ่นใหม่ (compute capability 12.1 / sm_121) ใหม่กว่าที่
# nvrtc ของ warp (CUDA 12.9) และ torch (cu128) รู้จัก (ทั้งคู่รองรับสูงสุด sm_120)
# ทำให้ JIT-compile CUDA kernel พังด้วย "invalid value for --gpu-architecture".
# แก้ด้วยการบังคับ warp สร้าง PTX เป็น sm_120 (sitecustomize) แล้วปล่อยให้ driver
# 13.0 JIT ต่อเป็น sm_121 ตอน runtime; ส่วน torch ปิด JIT fusion ด้วย
# PYTORCH_JIT=0 (ตั้งใน 2 cell ถัดไปตอนสั่งรัน)
import glob
sp = glob.glob(os.path.join(REPO, '.venv/lib*/python*/site-packages'))
_sc = ('# GB10 sm_121: force warp PTX to sm_120, driver JITs to sm_121.\n'
       'try:\n    import warp as _wp\n    _wp.config.ptx_target_arch = 120\n'
       'except Exception:\n    pass\n')
for _d in sp:
    with open(os.path.join(_d, 'sitecustomize.py'), 'w') as f:
        f.write(_sc)
print('ติดตั้ง sitecustomize (ptx_target_arch=120) ที่:', sp)
!cd {REPO} && {UV} run --no-sync python -c "import warp as wp; print('ptx_target_arch =', wp.config.ptx_target_arch)"

## 6) เทรน! (Lift-Cube — หยิบของบนโต๊ะ)

- fixed-base → ไม่มีปัญหาล้ม เรียนเร็วกว่า loco-manip มาก
- default 5000 iters (ปกติเห็นผลชัดตั้งแต่ ~2000-3000)
- GB10 VRAM เยอะ → num-envs 4096 ได้สบาย (ถ้า OOM ลดลง)

**ดู metrics:** `position_error` (มือ→cube) ควรลด, `lift` reward ควรเพิ่ม,
`episode_success` ควรขยับขึ้นจาก 0

In [ ]:
LOG_ROOT = os.path.join(WORKSPACE, 'mjlab_logs')
os.makedirs(LOG_ROOT, exist_ok=True)
# PYTORCH_JIT=0: ปิด torch JIT fusion (ไม่งั้น torch พยายาม compile fused kernel
# ตรงเป็น sm_121 ซึ่ง nvrtc cu128 ไม่รู้จัก) — จำเป็นสำหรับ GB10
!cd {REPO} && PYTORCH_JIT=0 {UV} run --no-sync python -m mjlab.scripts.train Mjlab-Lift-Cube-G1 \
    --env.scene.num-envs 4096 \
    --agent.max-iterations 5000 \
    --agent.save-interval 200 \
    --agent.logger tensorboard \
    --log-root {LOG_ROOT}

## 7) หาไฟล์โมเดล .pt

In [ ]:
from pathlib import Path
log_dir = Path(LOG_ROOT) / 'g1_lift_cube'
runs = sorted(log_dir.glob('*'), key=os.path.getmtime, reverse=True)
assert runs, 'ไม่พบ run — เทรนสำเร็จหรือยัง?'
ckpts = sorted(runs[0].glob('model_*.pt'), key=lambda p:int(''.join(filter(str.isdigit,p.stem))))
checkpoint = str(ckpts[-1])
print('✅ โมเดล:', checkpoint)

## 8) ทดสอบ — เรนเดอร์วิดีโอ policy ที่เทรนได้

เซฟวิดีโอลง workspace เช่นกัน

In [ ]:
os.environ.setdefault('MUJOCO_GL','egl')
VIDEO_PATH = os.path.join(LOG_ROOT, 'g1_lift_cube_spark.mp4')
render_script = f'''
import os; os.environ['MUJOCO_GL']='egl'
import torch, imageio
from dataclasses import asdict
import mjlab.tasks
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg, load_runner_cls
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import RslRlVecEnvWrapper
from mjlab.rl.runner import MjlabOnPolicyRunner
TASK='Mjlab-Lift-Cube-G1'
cfg=load_env_cfg(TASK, play=True); cfg.scene.num_envs=1
env=ManagerBasedRlEnv(cfg=cfg, device='cuda', render_mode='rgb_array')
ag=load_rl_cfg(TASK); rc=load_runner_cls(TASK) or MjlabOnPolicyRunner
w=RslRlVecEnvWrapper(env, clip_actions=ag.clip_actions)
r=rc(w, asdict(ag), device='cuda')
r.load(\'{checkpoint}\', load_cfg={{"actor":True}}, strict=True, map_location='cuda')
pol=r.get_inference_policy(device='cuda')
obs=w.get_observations(); frames=[]
for _ in range(300):
    with torch.inference_mode(): a=pol(obs)
    obs,_,_,_=w.step(a); frames.append(env.render())
imageio.mimsave(\'{VIDEO_PATH}\', frames, fps=30)
print('video ->', \'{VIDEO_PATH}\')
'''
open('/tmp/render_lc.py','w').write(render_script)
# PYTORCH_JIT=0 จำเป็นสำหรับ GB10 (sm_121) เช่นเดียวกับตอนเทรน
!cd {REPO} && PYTORCH_JIT=0 {UV} run --no-sync python /tmp/render_lc.py

In [ ]:
from IPython.display import Video
Video(VIDEO_PATH, embed=True, width=480)

## สรุป

โมเดล + วิดีโออยู่ใน `{WORKSPACE}/mjlab_logs/g1_lift_cube/` (storage เครื่อง ไม่หาย)

**เอาไปดูบน Mac:** ดาวน์โหลด .pt มา แล้ว
```bash
CUDA_VISIBLE_DEVICES='' uv run play Mjlab-Lift-Cube-G1 \
    --checkpoint-file model.pt --viewer viser --num-envs 1
```

**ขั้นต่อไป:** พอ Lift-Cube หยิบของได้ดีแล้ว ค่อยต่อยอดเป็น loco-manip
(`train_g1_locomanip_spark.ipynb`) โดยอาจใช้โมเดลนี้เป็นจุดตั้งต้นของการหยิบ